# 项目代码分析

## 数据处理

### 1. 读取数据集与标准化
- get_vars()
    return var_dict:dict

    从数据集中读取变量名与索引
    var_dict: {变量名: 索引列表}

- open_asm_merra(time:str)
    return total_matrix:ndarray, index:list, lon, lat, time_str

    从数据集中读取数据并下采样
    shape(total_matrix): (channel, latitude, longitude)

### 2. 标准化并转为 torch 张量

- get_init_field

    (数据 - 均值) / 标准差
    ```
    init_field = stand(index, total_matrix)
    ```

    转为 PyTorch 张量并添加 batch 维度，返回形状为 (1, 54, latitude, longitude)
    ```
    init_field = torch.from_numpy(init_field).to(torch.float16).unsqueeze(0).float()
    ```

    return init_field, init_time, index, lon, lat

    init_field: (1, 54, latitude, longitude):[B, C, H, W]

### 3. 调用模型
- output = rolling_model(init_field, init_time, index, lon, lat)
    return output: torch.Tensor

    模型输出，形状为 [B, 54, latitude, longitude]

### 4. 输出转换为 numpy 数组
- data_array, _ = convert2dataarray(output)
    return data_array: ndarray

    转换为 numpy 数组，形状为 (channel, latitude, longitude)


## 3h 小时模型模块分析

### 1. TransUnet

```text
输入：[B,54,H,W]

    │ 统一空间尺寸
    ▼

[B,54,576,576]

    │ 时间编码
    │ - 时间尺度归一化：hourofday, dayofweek, dayofyear
    │ - 周期特征编码：cos(2πt)、sin(2πt) 并 expand 为 [B, 6, 576, 576]
    │ - 将时间编码与通道编码合并, 形状为 [B, 60, 576, 576]
    ▼

[B,60,576,576]

    │ 卷积层
    │ - 卷积层1（Sequential）：[B, 60, 576, 576] → [B, 640, 288, 288]
    │ - 卷积层2（Bottleneck）：[B, 640, 288, 288] → [B, 1280, 144, 144]
    │ - 卷积层3（Bottleneck）：[B, 1280, 144, 144] → [B, 2560, 72, 72]
    │ - 卷积层4（Bottleneck）：[B, 2560, 72, 72] → [B, 5120, 36, 36]
    ▼

[B, 5120, 36, 36]

    │ Vit 层：[B, 5120, 36, 36] → [B, 324, 2400]，表示为 token
    ▼

[B, 324, 2400]

    │ project_patches_back: 将 token 线性投影为可还原为空间特征的向量
    ▼

[B, 324, 20480]

    │   - reshape: [B, 324, 20480] -> [B, 18, 18, 2, 2, 5120]
        - permute(0, 5, 3, 1, 4, 2): [B, 18, 18, 2, 2, 5120] -> [B, 5120, 2, 18, 2, 18]
        - reshape:[B, 5120, 2, 18, 2, 18] -> [B, 5120, 36, 36]
    ▼

[B, 5120, 36, 36]

    │ SingleConv
    ▼

[B, 2560, 36, 36]

    │ Up_1（dec1）：转置卷积上采样 → 跳跃连接拼接 → DoubleConv 融合
    │ - argument_1：[B, 2560, 36, 36]
    │ - argument_2（_30）：[B, 2560, 72, 72]
    │ - argument_1 上采样：[B, 2560, 72, 72]
    │ - 沿通道拼接：[B, 5120, 72, 72]
    │ - 卷积融合后输出：[B, 1280, 72, 72]
    ▼

[B, 1280, 72, 72]

    │ Up_2（dec2）：转置卷积上采样 → 跳跃连接拼接 → DoubleConv 融合
    │ - argument_1：[B, 1280, 72, 72]
    │ - argument_2（_29）：[B, 1280, 144, 144]
    │ - argument_1 上采样：[B, 1280, 144, 144]
    │ - 沿通道拼接：[B, 2560, 144, 144]
    │ - 卷积融合后输出：[B, 640, 144, 144]
    ▼

[B, 640, 144, 144]

    │ Up_3（dec3）：转置卷积上采样 → 跳跃连接拼接 → DoubleConv 融合
    │ - argument_1：[B, 640, 144, 144]
    │ - argument_2（_28）：[B, 640, 288, 288]
    │ - argument_1 上采样：[B, 640, 288, 288]
    │ - 沿通道拼接：[B, 1280, 288, 288]
    │ - 卷积融合后输出：[B, 320, 288, 288]
    ▼

[B, 320, 288, 288]

    │ dec4（Sequential）：双线性上采样＋卷积
    │ - 双线性上采样：[B, 320, 288, 288] → [B, 320, 576, 576]
    │ - 3×3 卷积将 320 个特征通道映射为 54 个预测通道
    ▼

[B, 54, 576, 576]

    │ 去掉前面添加的边界，再恢复输出网格
    │ - 裁去上下填充，高度恢复为原始输入高度 H：[B, 54, H, 576]
    │ - 裁去左右各 38 列周期填充：[B, 54, H, 500]
    │ - 最近邻重采样只将宽度调整为固定 576，高度保持 H
    ▼

[B, 54, H, 576]
```


### 2. Bottleneck

1. 卷积
2. 实例归一化
3. 残差相加
4. ReLU 引入非线性


### 3. ViT

```text
输入卷积后的特征
[B, 5120, 36, 36]
    │
    │ reshape：拆分空间维度
    ▼
[B, 5120, 2, 18, 2, 18]
    │
    │ permute(0, 3, 5, 2, 4, 1)：调整维度顺序
    │ 将四个象限中相对位置相同的四个点归入同一个 token
    │ 注意：这四个点不是相邻的 2×2 小块
    ▼
[B, 18, 18, 2, 2, 5120]
    │
    │ reshape：合并 token 位置维度和特征维度
    │ token 数量：18×18 = 324
    │ 每个 token 的特征维数：2×2×5120 = 20480
    ▼
[B, 324, 20480]
    │
    │ project_patches：线性投影，将 20480 维映射为 2400 维
    ▼
[B, 324, 2400]
    │
    │ 在序列最前面添加 cls_token
    │ cls_token 扩展后的形状：[B, 1, 2400]
    ▼
[B, 325, 2400]
    │
    │ 加上可学习的位置编码 pos_emb1D
    ▼
[B, 325, 2400]
    │
    │ embedding dropout：训练时随机置零部分元素，减少过拟合
    │ 不改变张量形状
    ▼
[B, 325, 2400]
    │
    │ 经过 14 层 TransformerBlock，让 token 之间交换信息
    ▼
[B, 325, 2400]
    │
    │ 删除序列中的第一个 token，即 cls_token
    ▼
输出空间 token 序列
[B, 324, 2400]
```

```python
import numpy as np

matrix = np.arange(36).reshape(6, 6)

# 拆分空间维度
arr = matrix.reshape(2, 3, 2, 3)
# print(arr)

# 调整维度顺序, 取四个象限中相同位置的四个点 
arr = arr.transpose(1, 3, 0, 2)

print(arr)


```


### 4. TransformerEncoder

14 层 TransformerBlock 层

### 5. TransformerBlock

```text
[B, 325, 2400]

    │ MultiHeadSelfAttention, 16 个多头注意力，交换 token 之间信息
    ▼

[B,325,2400]

    │ 残差相加，更新后的表示 = 原来的表示 + 根据全序列信息计算出的更新
    ▼

    │ 归一化
    ▼

    │ 每个 token 内部的特征进行组合和非线性变换
    ▼

    │ 残差相加，把前馈网络的输出加回来
    ▼

    │ 归一化
    ▼

[B, 325, 2400]
```


### 6. MultiHeadSelfAttention

```text
[B, 325, 2400]
    │
325 是 token 数量；2400 是每个 token 的特征维数；注意力头数为 16。
每头维数 d = 2400 / 16 = 150；Q/K/V 联合投影宽度为 3×2400 = 7200。
    ▼

    │ 生成 Q、K、V 所需的全部特征
    ▼

[B,325, 3 * 16 * d]
    │    16 个注意力头
    │ reshape: [B,325, 3 * 16 * d] -> [B, 325, d, 3, 16]
    ▼

[B, 325, d, 3, 16]

    │ permute: [B, 325, d, 3, 16] -> [3, B, 16, 325, d]
    ▼

[3, B, 16, 325, d]

    │ token 信息交换
    │ - 得到 Q、K、V， shape 都为 [B, 16, 325, d]
    │ - Q， K 点积后 softmax，[B, 16, 325, 325]
    │ - V 加权，[B, 16, 325, d]
    │ - permute: [B, 16, 325, d] -> [B, 325, 16, d]
    │ - 拼接，[B, 325, 16 * d]
    │ - 线性层混合各头的信息，[B, 325, 2400]
    ▼

[B, 325, 2400]
```

## 疑问

1. 代码选用了“500 列有效区域＋两侧各 38 列扩展＝576 列”的方案；为什么采用这组具体数值
2. 为什么同一 token 取四个象限中相对位置相同的点？若取相邻的点，是否会影响模型的训练？
3. cls_token 添加的作用是？是否对模型的训练有影响？
4. ___torch_mangle_3.Conv2d 和 ___torch_mangle_4.InstanceNorm2d 的作用和差异？
5. 上采样恢复空间时为什么要使用 卷积层3（Bottleneck）保存的特征而不是其他层？
    为了能够一步步上采样恢复空间



In [13]:
import numpy as np

matrix = np.arange(36).reshape(6, 6)

# 拆分空间维度
arr = matrix.reshape(2, 3, 2, 3)
# print(arr)

# 调整维度顺序, 取四个象限中相同位置的四个点 
arr = arr.transpose(1, 3, 0, 2)

print(arr)


[[[[ 0  3]
   [18 21]]

  [[ 1  4]
   [19 22]]

  [[ 2  5]
   [20 23]]]


 [[[ 6  9]
   [24 27]]

  [[ 7 10]
   [25 28]]

  [[ 8 11]
   [26 29]]]


 [[[12 15]
   [30 33]]

  [[13 16]
   [31 34]]

  [[14 17]
   [32 35]]]]


# 运行结果分析

[问题记录与排查计划](</Users/penpen/Public/git/AI-GAMFS/AI-GAMFS问题记录与排查计划.md>)，尤其是第 1 节、第 16 节及最后的区域验
目前已有的成果包括：

- 单一起报的 120 小时预报、40 份输出，结构和有限值检查通过。
- 同目标时刻的路径与输入精度对照。
- 与论文案例参考场的比较，单案例表现总体接近。
- 原始 MERRA-2 的单点及 21×21 区域配对，单位、坐标、残差符号和指标计算已验证。

**现在的主要缺口是独立日期覆盖，而不是模型能否运行。** 文档仍记录为 1 个独立起报、0 个可训练案例；区域内的 441 个格点不能替代多个独立天气过程。

建议按下面的顺序推进。

**第一步：把近期研究问题缩小。**

暂时先问：

> AI-GAMFS 的气溶胶预报误差，是否存在跨日期稳定、能由简单方法校正的系统性偏差？

先选一个主要变量，例如你已有较多诊断的 `OCSMASS`，并固定少量时效，例如 +12h、+24h、+72h。其他变量可以保留作辅助分析。这样更容易完成第一轮实验并解释结果。

**第二步：先设计多日期预演，再扩大数据。**

下一份具体产出应是一张“候选日期与数据可得性表”，包括起报日期、输入是否齐全、参考数据是否可读、预报时效、时间窗口口径和预计存储量。

可以先用几个相隔开的起报日期验证批处理流程。这一小批只用于发现数据与流程问题，不能据此证明方法具有普遍效果。正式实验再按季节、事件覆盖和预算扩展，并按日期块划分训练、验证和测试；跨划分还要避免预报有效时间重叠。

文档中尚未确认的**基线输出平均窗口**需要继续保留为限制。先明确采用的参考目标与评价口径，不要把小时参考场与模型输出的时间语义默认为完全一致。

**第三步：先做简单校正基线。**

具备多日期配对后，依次比较：

| 方法 | 要回答的问题 |
|---|---|
| 原始预报 | 基础误差有多大？ |
| 输出后非负截断 | 仅满足非负约束能改变多少误差？ |
| 训练期平均偏差校正 | 误差是否存在稳定的均值偏差？ |
| 轻量残差模型 | 条件信息是否提供额外收益？ |

所有拟合参数都只从训练期获得。先看独立日期上的 Bias、MAE、纬度加权 RMSE，再决定是否进入 score 或扩散模型。文档已明确“残差不等于 score”，这一区分应继续保留。

至于**什么时候需要复原模型代码**，取决于研究对象：

| 你要做的研究 | 是否需要完整复原 |
|---|---|
| 分析预报误差、比较接力路径 | 不需要 |
| 在冻结预报后训练校正模型 | 不需要 |
| 学习注意力、卷积等模块 | 可以单独实现小模块 |
| 修改 token 划分、移除 cls token、改变网络或微调主模型 | 需要可训练模型定义及权重映射 |

如果以后选择修改主模型，复原代码也要分两步：先在相同输入、时间标签和推理设置下，验证复原模型与 TorchScript 输出数值一致；再修改结构。**复原前向计算仍不等于复现原训练过程**，后者还需要数据构建、损失函数、训练配置和选模规则。

对你现在的六个月计划，我建议把接下来一周的成果定为：**完成一页多日期实验设计，确定变量、时效、参考目标、候选日期、划分规则和比较方法。** 完整模型复原可以留作后续架构研究的支线，暂时不必成为进入科研实验的前置条件。